# Capstone — Deployed Research Paper (FlyRank Content Refresh Lane)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliHamza-lab/Ml_intership/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This notebook collects the sections for the FlyRank refresh-opportunity lane. The case study is the practical problem of helping a content team decide which existing pages deserve a refresh review when editorial time is limited and search visibility is fading.

## 0. Title & Abstract

**Title:** Data-Driven Editorial Triage: Prioritizing Content Refresh Interventions via Learned Rankers on Real Production Search Data

**Abstract:**
In digital publishing operations, content teams struggle to decide which of their thousands of existing pages should be prioritized for updates when search visibility begins to decline. This paper frames a public-safe content triage problem using an anonymized dataset of 30,000 production pages. We construct a transparent baseline rule scoring system and compare it to three learned classifiers under a client-holdout validation design to prevent data leakage. The Random Forest classifier achieves the best performance with a Test ROC AUC of 0.747 and Average Precision of 0.610, delivering a 2.8x lift in Precision@50 (0.680 vs. 0.240) over the baseline rules. The final output is formatted as a prioritized queue and action playbook to support editorial decisions under resource constraints.

## 1. Introduction / Problem Statement

For large-scale digital publishers, refreshing existing content is one of the most efficient growth and maintenance levers. However, editorial budgets and writer hours are strictly limited, making it impossible to manually audit every page. Reviewing high-traffic pages that show signs of performance decay (fading search visibility, low click-through rate, or dropping engagement) is critical. 

The core decision this supports is **editorial triage**: which pages should be sent to a reviewer for updates? 
- **Unit of analysis:** Individual page/content item.
- **Output:** Prioritization score (0-100), suggested review action, and confidence tier.
- **Action:** A content editor checks the page against recent search intent to update, expand, or modify it.
- **Cost of wrong calls:** 
  - *False Positive (recommending a healthy page):* Operational waste. Editors spend time revising content that doesn't need it, potentially destabilizing successful rankings.
  - *False Negative (missing a declining page):* Opportunity cost. The page continues to lose traffic, impressions, and revenue to competitors.

In [1]:
# Find repo root and load the prepared feature vector
import os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Locate root dir by searching for requirements.txt
root = Path(os.getcwd())
for _ in range(5):
    if (root / 'requirements.txt').exists():
        break
    root = root.parent

print("Detected repository root:", root)
features_path = root / 'data' / 'processed' / 'refresh_feature_vector.csv'
df = pd.read_csv(features_path)
print(f"Loaded dataset with {df.shape[0]:,} rows and {df.shape[1]} columns.")

Detected repository root: d:\Flyrank_internship\Ml_intership


Loaded dataset with 30,000 rows and 52 columns.


## 2. Data

We use the anonymized starter dataset (`data/raw/content_refresh_anonymized.csv`), which covers 30,000 pages of performance history. The dataset is fully public-safe: it contains no client names, domains, URLs, page titles, or raw search queries. Hashed `content_id` and `client_id` represent pseudonyms.

**Exclusions and Rationale:**
1. **`impressions_90d > 0`**: We exclude pages with zero search impressions in the last 90 days. If a page has no search visibility, there is no performance trend to evaluate or save through a refresh.
2. **`content_age_days >= 90`**: We exclude pages published for less than 90 days. Fresh pages are in their initial indexation/ranking phase, where volatility is normal and a refresh review is premature.
3. **Duplicates on `content_id`** are removed to ensure each content page appears exactly once.

In [2]:
# Show data characteristics and exclusions stats
raw_path = root / 'data' / 'raw' / 'content_refresh_anonymized.csv'
raw_df = pd.read_csv(raw_path)

print(f"Raw dataset rows: {len(raw_df):,}")
print(f"Excluded due to zero impressions: {(raw_df['impressions_90d'] <= 0).sum():,}")
print(f"Excluded due to age < 90 days: {(raw_df['content_age_days'] < 90).sum():,}")
print(f"Final prepared feature vector rows: {len(df):,}")
print(f"Decline label base rate: {df['is_declining_label'].mean():.3%}")

Raw dataset rows: 30,000
Excluded due to zero impressions: 0
Excluded due to age < 90 days: 0
Final prepared feature vector rows: 30,000
Decline label base rate: 54.207%


## 3. Methodology

### Assumptions
- Pages with high visibility (high impressions) and low relative performance (low CTR/sessions) represent the most valuable triage opportunities.
- Historical performance trends of a client's pages are correlated, so we must validate across distinct clients.

### Label Definition
The target label is `is_declining_label`, defined as `trend_direction == 'down'` (indicating search traffic loss over the comparison windows). Because the label is derived directly from this direction, `trend_direction` and `trend_pct` are strictly excluded from the feature set to prevent label leakage.

### Validation Design & Leakage Checks
We use a **Grouped Client-Holdout split** (80% train / 20% test based on `client_id`). This ensures that no individual client's pages appear in both the training and testing sets, providing an honest evaluation of how the model generalizes to new sites. We also ensure that no client-identifying pseudonyms or metadata are used as training features.

### Baseline
The baseline is a deterministic rule-based score (`baseline_refresh_score`) calculated as:
- **40%** Visibility Score (log impressions percentile)
- **30%** Freshness Risk Score (days since last update percentile)
- **25%** Position Opportunity Score (favorable ranking tier percentile × visibility)
- **5%** Depth Gap Score (1 - word count percentile × visibility)

In [3]:
# Load pipeline configurations and feature dimensions
results_path = root / 'outputs' / 'model_results.json'
with open(results_path, 'r') as f:
    results = json.load(f)

print("Validation split strategy:", results['split_strategy'])
print(f"Training set rows: {results['train_rows']:,}")
print(f"Testing set rows: {results['test_rows']:,}")
print(f"Feature count: {results['feature_count']}")
print("Numeric features:", results['model_numeric_features'])
print("Categorical features:", results['model_categorical_features'])

Validation split strategy: client_holdout
Training set rows: 27,675
Testing set rows: 2,325
Feature count: 52
Numeric features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
Categorical features: ['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']


## 4. Results (vs Baseline)

We trained three classifiers on the client-holdout training set and evaluated them on the test set. The models compared are: Logistic Regression (with balanced class weights), a shallow Decision Tree, and a Random Forest Classifier.

In [4]:
# Model comparison table
models_dict = results['models']
baseline_dict = results['baseline']

comparison = []
for m_name, m_metrics in models_dict.items():
    comparison.append({
        'Model': m_name,
        'ROC AUC': m_metrics['roc_auc'],
        'Avg Precision': m_metrics['average_precision'],
        'Precision@50': m_metrics['precision_at_50'],
        'Recall': m_metrics['recall'],
        'F1': m_metrics['f1']
    })
comparison.append({
    'Model': 'baseline_rules',
    'ROC AUC': baseline_dict['baseline_roc_auc'],
    'Avg Precision': baseline_dict['baseline_average_precision'],
    'Precision@50': baseline_dict['baseline_precision_at_50'],
    'Recall': baseline_dict['baseline_recall'],
    'F1': baseline_dict['baseline_f1']
})

comp_df = pd.DataFrame(comparison)
print(comp_df.to_string(index=False))

              Model  ROC AUC  Avg Precision  Precision@50   Recall       F1
      decision_tree 0.741520       0.575319          0.62 0.716172 0.633885
logistic_regression 0.700291       0.521542          0.40 0.566557 0.566245
      random_forest 0.747372       0.610058          0.68 0.741474 0.638258
     baseline_rules 0.626892       0.467607          0.24 0.189219 0.274322


### Key Takeaways
- The **Random Forest** model achieves the highest test metrics with an **ROC AUC of 0.747** and **Average Precision of 0.610**.
- Under a baseline rule model, the Precision@50 is **0.240**, meaning only 24% of the top 50 prioritized pages are actually declining. The Random Forest model yields a **Precision@50 of 0.680**, representing a **2.83× lift** (68% vs 24%). This demonstrates the clear benefit of using a learned ranker over simple heuristic scores.
- Feature importances indicate that `days_with_impressions` (16.06%), `log_impressions_90d` (12.85%), and `avg_position` (10.84%) are the primary indicators of content decline risk.

## 5. Limitations & Honest Framing

We present these findings with the following limits:
1. **Single Labeled Snapshot:** The data represents a static historical snapshot. We cannot guarantee that these patterns will hold indefinitely as search algorithms and user behaviors shift.
2. **Causal Claims Limitation:** The model identifies correlations between page features and performance decline. We do not claim that having high impressions *causes* decline, or that editing a page will *guarantee* recovery. The outputs must be treated as decision-support aids.
3. **Lack of Live Testing:** The models are validated on historical test splits. A true measure of the model's impact would require a randomized controlled trial (A/B test) in a live production environment.

## 6. Ranked Recommendations

We blend the Random Forest predictions (70% weight) with the normalized baseline rules score (30% weight) to create the final `final_refresh_score` (scaled 0-100). This blend ensures pages are ranked by both their statistical decline risk and their operational visibility/relevance. 

### Suggested Editorial Actions
- **`refresh_and_review_ctr`**: For pages with high search impressions but low click-through rates. Editors should focus on metadata, titles, and search snippet optimization.
- **`refresh_and_review_engagement`**: For visible pages with high clicks but dropping sessions, scroll rates, or engagement. Editors should review search intent match, content readability, and call-to-actions.
- **`expand_and_refresh`**: For thin pages (low word counts) that still show visible search demand. Editors should expand the text and add depth.
- **`refresh`**: General content updates for stale pages with high decline probability.
- **`monitor`**: Pages that do not meet high-risk thresholds.

In [5]:
# Load final queue and print suggested actions mix
queue_path = root / 'outputs' / 'refresh_queue.csv'
queue = pd.read_csv(queue_path)
print("Suggested Action Counts:")
print(queue['suggested_action'].value_counts())
print("\nConfidence Tier Counts:")
print(queue['confidence'].value_counts())
print("\nTop 10 items in the queue preview:")
print(queue[['final_rank', 'content_id', 'final_refresh_score', 'suggested_action', 'best_model_probability', 'impressions_90d', 'ctr']].head(10).to_string(index=False))

Suggested Action Counts:
suggested_action
monitor                          13069
refresh                           8207
refresh_and_review_ctr            6655
refresh_and_review_engagement     1987
expand_and_refresh                  82
Name: count, dtype: int64

Confidence Tier Counts:
confidence
low       15000
medium    11424
high       3576
Name: count, dtype: int64

Top 10 items in the queue preview:
 final_rank           content_id  final_refresh_score       suggested_action  best_model_probability  impressions_90d  ctr
          1 content_1f080331fa2b            81.928467 refresh_and_review_ctr                0.786247            12834 0.05
          2 content_6aa43079fb0c            81.728449 refresh_and_review_ctr                0.792117             8064 0.07
          3 content_d6570c51c9bd            81.639118 refresh_and_review_ctr                0.850354             2498 0.00
          4 content_e04eb9549989            80.804986 refresh_and_review_ctr                0.81383

## 7. Artifacts the Paper Embeds

The pipeline generates charts as SVGs in the outputs folder. Let's inspect the files.

In [6]:
charts_dir = root / 'outputs' / 'charts'
print(f"Charts directory: {charts_dir.resolve()}")
for chart_file in charts_dir.glob('*.svg'):
    print(f"- Found chart: {chart_file.name} ({chart_file.stat().st_size:,} bytes)")

Charts directory: D:\Flyrank_internship\Ml_intership\outputs\charts
- Found chart: action_mix.svg (1,698 bytes)
- Found chart: confidence_mix.svg (1,100 bytes)
- Found chart: top_feature_importance.svg (3,646 bytes)
- Found chart: top_reason_codes.svg (3,469 bytes)
- Found chart: trend_distribution.svg (1,649 bytes)


## 8. Reproducibility

To reproduce this analysis from a fresh clone, run the following steps:
1. Ensure you have Python installed.
2. Install dependencies: `pip install -r requirements.txt`
3. Run the full execution pipeline: `python scripts/run_all.py`
4. Verify the outputs in the `outputs/` folder.

### Random Seed
A random seed of `42` is fixed in all scripts (`scripts/02_baseline_score.py`, `scripts/03_train_model.py`) to guarantee consistent split indexing and model initialization.

## 9. Acknowledgments & Data Credit

This work is built on the FlyRank ML Internship dataset. We thank the team at [FlyRank](https://flyrank.ai) for providing the anonymized production search dataset that makes this research possible.

## ML-12 Closing Section

### 5-Minute Showcase Demo Outline
1. **The Question (1 min):** Explain the editorial triage problem: how to prioritize pages for review under tight content budgets.
2. **The Method (1 min):** Discuss the anonymized 30,000-page dataset, validation split strategy (grouped client-holdout to avoid leakage), and the baseline heuristic scoring rule.
3. **The Results (1.5 min):** Contrast the baseline Precision@50 (24%) with the Random Forest Classifier's Precision@50 (68%), demonstrating a 2.8x lift in identifying declining pages.
4. **The Recommendation (1 min):** Walk through the Action Playbook: translating the ranked queue into targeted review activities (`refresh_and_review_ctr`, `refresh_and_review_engagement`, etc.).
5. **Q&A (0.5 min):** Conclude with honest limitations regarding correlation vs causality and the lack of live production testing.

### Shareable Cuts
**Short Social Post:**
I've built an ML-based content triage system using 30,000 rows of anonymized search data! By validating with a grouped client-holdout design, the Random Forest model achieved a 2.8x lift in Precision@50 (68% vs 24%) over heuristic rules, translating model risks into concrete editorial refresh actions. Check it out: https://alihamza-lab.github.io/Ml_intership/

**Employer-Facing Summary:**
Developed a decision-support ranker for editorial content refreshes based on a 30,000-page anonymized production search dataset. Designed a grouped client-holdout cross-validation framework to prevent data leakage. Implemented a Random Forest Classifier that delivered a 2.8x lift in Precision@50 (0.680 vs. 0.240) over the baseline rules. Output is delivered as an actionable priority queue with automated review recommendations based on specific page traffic decay patterns.